# Notebook 05 — Agent Inference Demo

This notebook is the final inference layer for the **Dating Red Flag Detector: Interactive Profile Auditing Agent**.

It loads the artifacts produced by Notebooks 01–04, selects one OkCupid profile, runs the trained NLP and tabular models, attaches the YOLO visual result, combines everything into one structured assessment, and then sends that assessment to OpenRouter for a polished LLM-style report.

Notebook 05 is therefore an **inference/demo notebook**, not another training notebook.

## 1. Imports, paths, and configuration

Run this only after Notebooks 01–04 have produced the cleaned dataset, saved models, visual features, and metric CSVs.

In [1]:
import os
import json
import html
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests

from IPython.display import display, Markdown, HTML

warnings.filterwarnings("ignore")

try:
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    TRANSFORMERS_AVAILABLE = True
except Exception as e:
    TRANSFORMERS_AVAILABLE = False
    TRANSFORMER_IMPORT_ERROR = str(e)


def find_project_root():
    # Works whether this notebook is opened from the project root or from a /notebooks folder.
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "data").exists() or (candidate / "models").exists() or (candidate / "reports").exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

PATHS = {
    "cleaned_data": DATA_DIR / "okcupid_cleaned_redflags.csv",
    "nlp_model": MODELS_DIR / "nlp_tfidf_logreg_model.pkl",
    "nlp_labels": MODELS_DIR / "nlp_label_columns.pkl",
    "nlp_thresholds": MODELS_DIR / "nlp_custom_thresholds.pkl",
    "tabular_models": MODELS_DIR / "tabular_rf_smote_models.pkl",
    "tabular_preprocessor": MODELS_DIR / "tabular_preprocessor.pkl",
    "visual_features": DATA_DIR / "visual_inference_features.csv",
    "visual_metrics": REPORTS_DIR / "yolo_visual_metrics.csv",
    "nlp_metrics": REPORTS_DIR / "nlp_overall_metrics.csv",
    "tabular_metrics": REPORTS_DIR / "tabular_overall_metrics.csv",
}

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Models directory:", MODELS_DIR)
print("Reports directory:", REPORTS_DIR)

Project root: C:\Users\LENOVO\anaconda3\Red-Flagger
Data directory: C:\Users\LENOVO\anaconda3\Red-Flagger\data
Models directory: C:\Users\LENOVO\anaconda3\Red-Flagger\models
Reports directory: C:\Users\LENOVO\anaconda3\Red-Flagger\reports


## 2. Load dataset, trained models, and previous reports

If this cell raises a missing-file error, rerun the previous notebook that creates that artifact.

In [2]:
required_files = [
    "cleaned_data",
    "nlp_model",
    "nlp_labels",
    "nlp_thresholds",
    "tabular_models",
    "tabular_preprocessor",
]

missing = [name for name in required_files if not PATHS[name].exists()]
if missing:
    raise FileNotFoundError(
        "Missing required files: " + ", ".join(missing) +
        "\nRun Notebooks 01, 02, and 03 before running Notebook 05."
    )

df = pd.read_csv(PATHS["cleaned_data"])

nlp_model = joblib.load(PATHS["nlp_model"])
label_columns = joblib.load(PATHS["nlp_labels"])
nlp_thresholds = joblib.load(PATHS["nlp_thresholds"])

tabular_models = joblib.load(PATHS["tabular_models"])
tabular_preprocessor = joblib.load(PATHS["tabular_preprocessor"])

visual_df = pd.read_csv(PATHS["visual_features"]) if PATHS["visual_features"].exists() else pd.DataFrame()
visual_metrics_df = pd.read_csv(PATHS["visual_metrics"]) if PATHS["visual_metrics"].exists() else pd.DataFrame()
nlp_metrics_df = pd.read_csv(PATHS["nlp_metrics"]) if PATHS["nlp_metrics"].exists() else pd.DataFrame()
tabular_metrics_df = pd.read_csv(PATHS["tabular_metrics"]) if PATHS["tabular_metrics"].exists() else pd.DataFrame()

print("Cleaned dataset shape:", df.shape)
print("NLP labels:", label_columns)
print("Visual features loaded:", not visual_df.empty)
print("Visual metrics loaded:", not visual_metrics_df.empty)

Cleaned dataset shape: (700, 30)
NLP labels: ['aggressive_tone', 'hookup_focus', 'negativity', 'sarcasm_cynicism', 'substance_risk', 'incomplete_profile']
Visual features loaded: True
Visual metrics loaded: True


## 3. Label descriptions and profile fields

These make the report easier to understand and reduce how much the LLM has to infer.

In [3]:
label_descriptions = {
    "aggressive_tone": "Harsh, hostile, demanding, or insulting wording in the bio.",
    "hookup_focus": "Strong focus on casual hookups or non-committal dating intent.",
    "negativity": "Frequent pessimistic, cynical, or emotionally negative statements.",
    "sarcasm_cynicism": "Sarcastic or dismissive tone that may need careful interpretation.",
    "substance_risk": "Signals of heavy drinking, smoking, drugs, or party-heavy lifestyle.",
    "incomplete_profile": "Low-effort profile with limited useful self-description.",
}

visual_flag_descriptions = {
    "clean": "No major visual presentation issue detected.",
    "No_Face_Present": "The image may not show a clear person/face, making identity unclear.",
    "Group_Photo_Ambiguity": "The image contains multiple people, making it unclear who owns the profile.",
    "Face_Obscured": "The face appears hidden, covered, filtered, or visually unclear.",
}

tabular_features = [
    "age", "status", "sex", "orientation", "body_type", "diet", "drinks", "drugs",
    "education", "ethnicity", "height", "income", "job", "offspring", "pets",
    "religion", "sign", "smokes"
]

profile_display_fields = [
    "age", "status", "sex", "orientation", "body_type", "diet", "drinks", "drugs",
    "education", "job", "offspring", "pets", "religion", "smokes"
]

## 4. Select one OkCupid profile

By default, this chooses a random row that has a live YOLO visual result when available. That makes the demo stronger because the visual section can show IoU. Set `PREFER_LIVE_YOLO_SAMPLE = False` for a completely random OkCupid row.

In [4]:
SEED = 42
PREFER_LIVE_YOLO_SAMPLE = True
rng = np.random.default_rng(SEED)

if PREFER_LIVE_YOLO_SAMPLE and not visual_df.empty and "visual_flag_source" in visual_df.columns:
    live_ids = visual_df.loc[visual_df["visual_flag_source"] == "yolo_live", "profile_id"].astype(int).tolist()
    candidate_ids = [idx for idx in live_ids if 0 <= idx < len(df)]
else:
    candidate_ids = list(range(len(df)))

if not candidate_ids:
    candidate_ids = list(range(len(df)))

SAMPLE_PROFILE_ID = int(rng.choice(candidate_ids))
sample_profile = df.iloc[SAMPLE_PROFILE_ID].to_dict()

print("Selected profile_id:", SAMPLE_PROFILE_ID)

display(pd.DataFrame({
    "field": profile_display_fields,
    "value": [sample_profile.get(field, "") for field in profile_display_fields]
}))

bio_preview = str(sample_profile.get("full_bio", ""))[:1500]
display(Markdown("### Bio preview\n" + (bio_preview if bio_preview.strip() else "_No bio text available._")))

Selected profile_id: 0


,field,value
0,age,26
1,status,single
2,sex,m
3,orientation,straight
4,body_type,average
5,diet,anything
6,drinks,socially
7,drugs,never
8,education,graduated from high school
9,job,other


### Bio preview
i am really obsessed with music and would love to do something with it. my music idol is tom delonge. my friends always make fun of me because i buy anything he does. i am supervisor and i am really bored of it. i really need to get back into school. guitar i like to think i am alright. i am always trying to improve at it. well used to be my hair. had to get rid of the blonde mohawk. so now i would say my plugs. i usually get somekind of comment to my earrings. to many movies to just pick one and same with tv shows. but some of the best are scott pilgrim and the office. music is the best thing ever and i surround myself with it everyday. pizza is the best food i could eat it when ever. my brother my guitars my friends my phone my ps3 and oxygen obviously  trying to have fun in what ever is fun to do that night.  want to be friends. to see whats up? or what ever reason you can think of. lets talk

## 5. Inference functions

These functions reuse the saved models from Notebook 02 and Notebook 03, and the exported visual features from Notebook 04.

In [5]:
def run_nlp_inference(profile_text):
    classifier = nlp_model.named_steps["classifier"]
    tfidf = nlp_model.named_steps["tfidf"]
    text_features = tfidf.transform([str(profile_text)])

    results = {}
    detected_flags = []

    for label, estimator in zip(label_columns, classifier.estimators_):
        probability = estimator.predict_proba(text_features)[0][1]
        threshold = nlp_thresholds.get(label, 0.5)
        predicted = int(probability >= threshold)

        results[label] = {
            "probability": round(float(probability), 4),
            "threshold": float(threshold),
            "predicted": predicted,
            "description": label_descriptions.get(label, "")
        }
        if predicted:
            detected_flags.append(label)

    return {
        "module": "TF-IDF Logistic Regression NLP model",
        "detected_flags": detected_flags,
        "label_scores": results
    }


def run_transformer_inference(profile_text):
    model_path = MODELS_DIR / "distilbert_redflag_model"

    if not TRANSFORMERS_AVAILABLE:
        return {"available": False, "reason": "transformers/torch is not available.", "detected_flags": [], "label_scores": {}}

    if not model_path.exists():
        return {"available": False, "reason": "Saved DistilBERT folder was not found. Rerun Notebook 02 transformer section if needed.", "detected_flags": [], "label_scores": {}}

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.eval()

    inputs = tokenizer(str(profile_text), return_tensors="pt", truncation=True, padding=True, max_length=256)
    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.sigmoid(outputs.logits).numpy()[0]

    results = {}
    detected_flags = []
    for label, probability in zip(label_columns, probabilities):
        predicted = int(probability >= 0.5)
        results[label] = {
            "probability": round(float(probability), 4),
            "threshold": 0.5,
            "predicted": predicted,
            "description": label_descriptions.get(label, "")
        }
        if predicted:
            detected_flags.append(label)

    return {"available": True, "module": "DistilBERT Transformer NLP model", "detected_flags": detected_flags, "label_scores": results}


def run_tabular_inference(profile_row):
    input_data = {feature: profile_row.get(feature, np.nan) for feature in tabular_features}
    input_df = pd.DataFrame([input_data])
    processed_input = tabular_preprocessor.transform(input_df)

    results = {}
    detected_flags = []

    for label, model in tabular_models.items():
        probability = model.predict_proba(processed_input)[0][1]
        predicted = int(probability >= 0.5)
        results[label] = {
            "probability": round(float(probability), 4),
            "threshold": 0.5,
            "predicted": predicted,
            "description": label_descriptions.get(label, "")
        }
        if predicted:
            detected_flags.append(label)

    return {"module": "SMOTE Random Forest tabular model", "detected_flags": detected_flags, "label_scores": results}


def safe_json_loads(value, fallback):
    try:
        if pd.isna(value):
            return fallback
        return json.loads(value)
    except Exception:
        return fallback


def get_visual_assessment(profile_id):
    if visual_df.empty:
        return {"available": False, "reason": "visual_inference_features.csv was not found. Run Notebook 04 first.", "detected_visual_flags": [], "visual_risk_score": 0.0}

    match = visual_df[visual_df["profile_id"].astype(int) == int(profile_id)]
    if match.empty:
        return {"available": False, "reason": f"No visual row found for profile_id {profile_id}.", "detected_visual_flags": [], "visual_risk_score": 0.0}

    row = match.iloc[0].to_dict()
    flags = [flag for flag in str(row.get("detected_visual_flags", "clean")).split("|") if flag]

    visual_risk_lookup = {
        "clean": 0.0,
        "Face_Obscured": 0.65,
        "Group_Photo_Ambiguity": 0.70,
        "No_Face_Present": 0.80,
    }
    visual_risk_score = max([visual_risk_lookup.get(flag, 0.5) for flag in flags] or [0.0])

    return {
        "available": True,
        "module": "YOLOv8 visual heuristic module",
        "detected_visual_flags": flags,
        "visual_risk_score": round(float(visual_risk_score), 4),
        "visual_flag_explanations": {flag: visual_flag_descriptions.get(flag, "") for flag in flags},
        "bounding_boxes": safe_json_loads(row.get("bounding_boxes"), []),
        "confidence_scores": safe_json_loads(row.get("confidence_scores"), []),
        "num_persons_detected": int(row.get("num_persons_detected", 0)) if not pd.isna(row.get("num_persons_detected", np.nan)) else 0,
        "iou_metric": None if pd.isna(row.get("iou_metric", np.nan)) else round(float(row.get("iou_metric")), 4),
        "visual_flag_source": row.get("visual_flag_source", "unknown"),
        "image_path": row.get("image_path", "none")
    }

## 6. Ensemble auditor

The auditor combines model probabilities into a single risk score and keeps all raw module outputs for explanation.

In [6]:
def combine_label_scores(nlp_result, tabular_result, transformer_result=None):
    combined = {}

    for label in label_columns:
        scores = []
        predictions = []

        for result in [nlp_result, tabular_result, transformer_result]:
            if result and result.get("label_scores") and label in result["label_scores"]:
                scores.append(result["label_scores"][label]["probability"])
                predictions.append(result["label_scores"][label]["predicted"])

        mean_probability = float(np.mean(scores)) if scores else 0.0
        max_probability = float(np.max(scores)) if scores else 0.0
        predicted = int(mean_probability >= 0.5 or any(predictions))

        combined[label] = {
            "mean_probability": round(mean_probability, 4),
            "max_probability": round(max_probability, 4),
            "ensemble_predicted": predicted,
            "description": label_descriptions.get(label, "")
        }

    return combined


def risk_level(score):
    if score >= 70:
        return "High"
    if score >= 40:
        return "Moderate"
    if score >= 20:
        return "Low-to-Moderate"
    return "Low"


def load_metrics_summary():
    summary = {}

    if not nlp_metrics_df.empty:
        summary["nlp_tfidf_logreg_overall"] = nlp_metrics_df.to_dict(orient="records")

    if not tabular_metrics_df.empty:
        summary["tabular_smote_random_forest_overall"] = tabular_metrics_df.to_dict(orient="records")

    if not visual_metrics_df.empty:
        summary["yolo_visual_subset"] = {
            "mean_precision": round(float(visual_metrics_df["precision"].mean()), 4) if "precision" in visual_metrics_df.columns else None,
            "mean_recall": round(float(visual_metrics_df["recall"].mean()), 4) if "recall" in visual_metrics_df.columns else None,
            "mean_iou": round(float(visual_metrics_df["mean_iou"].mean()), 4) if "mean_iou" in visual_metrics_df.columns else None,
            "rows": visual_metrics_df.to_dict(orient="records")
        }

    return summary


def build_agent_assessment(profile_id):
    profile_row = df.iloc[profile_id].to_dict()
    profile_text = profile_row.get("clean_bio", profile_row.get("full_bio", ""))

    nlp = run_nlp_inference(profile_text)
    transformer = run_transformer_inference(profile_text)
    tabular = run_tabular_inference(profile_row)
    visual = get_visual_assessment(profile_id)

    combined_scores = combine_label_scores(nlp, tabular, transformer)
    detected_text_tabular_flags = [label for label, info in combined_scores.items() if info["ensemble_predicted"] == 1]

    label_probabilities = [info["mean_probability"] for info in combined_scores.values()]
    top_label_risk = float(np.mean(sorted(label_probabilities, reverse=True)[:3])) if label_probabilities else 0.0
    visual_risk = visual.get("visual_risk_score", 0.0)
    overall_score = round((0.70 * top_label_risk + 0.30 * visual_risk) * 100, 2)

    return {
        "profile_id": int(profile_id),
        "profile_summary": {field: profile_row.get(field, None) for field in profile_display_fields},
        "bio_excerpt": str(profile_row.get("full_bio", ""))[:1200],
        "overall_risk_score": overall_score,
        "overall_risk_level": risk_level(overall_score),
        "detected_text_tabular_flags": detected_text_tabular_flags,
        "detected_visual_flags": visual.get("detected_visual_flags", []),
        "combined_label_scores": combined_scores,
        "module_outputs": {
            "nlp_tfidf_logreg": nlp,
            "nlp_transformer": transformer,
            "tabular_smote_random_forest": tabular,
            "visual_yolo": visual,
        },
        "previous_model_performance": load_metrics_summary()
    }

raw_assessment = build_agent_assessment(SAMPLE_PROFILE_ID)
print(json.dumps(raw_assessment, indent=2)[:6000])

Loading weights: 100%|█████████████████████████████████████████████████████████████| 104/104 [00:00<00:00, 2499.36it/s]


{
  "profile_id": 0,
  "profile_summary": {
    "age": 26,
    "status": "single",
    "sex": "m",
    "orientation": "straight",
    "body_type": "average",
    "diet": "anything",
    "drinks": "socially",
    "drugs": "never",
    "education": "graduated from high school",
    "job": "other",
    "offspring": NaN,
    "pets": NaN,
    "religion": NaN,
    "smokes": "no"
  },
  "bio_excerpt": "i am really obsessed with music and would love to do something with it. my music idol is tom delonge. my friends always make fun of me because i buy anything he does. i am supervisor and i am really bored of it. i really need to get back into school. guitar i like to think i am alright. i am always trying to improve at it. well used to be my hair. had to get rid of the blonde mohawk. so now i would say my plugs. i usually get somekind of comment to my earrings. to many movies to just pick one and same with tv shows. but some of the best are scott pilgrim and the office. music is the best thing 

## 7. Display previous model metrics

This is where Notebook 05 satisfies the assignment requirement to mention Precision, Recall, and YOLO IoU in the final audit report.

In [7]:
if not nlp_metrics_df.empty:
    display(Markdown("### NLP overall metrics"))
    display(nlp_metrics_df)

if not tabular_metrics_df.empty:
    display(Markdown("### Tabular SMOTE overall metrics"))
    display(tabular_metrics_df)

if not visual_metrics_df.empty:
    display(Markdown("### YOLO visual metrics"))
    display(visual_metrics_df)

if nlp_metrics_df.empty and tabular_metrics_df.empty and visual_metrics_df.empty:
    print("No metric CSV files found yet. Rerun Notebooks 02–04 if you want metrics attached to the final report.")

### NLP overall metrics

,Metric,Score
0,Micro Precision,0.489796
1,Micro Recall,0.410256
2,Micro F1,0.446512
3,Macro Precision,0.346751
4,Macro Recall,0.326767
5,Macro F1,0.309200
6,Hamming Loss,0.141667


### Tabular SMOTE overall metrics

,Metric,Score
0,Micro Precision,0.860000
1,Micro Recall,0.367521
2,Micro F1,0.514970
3,Macro Precision,0.159259
4,Macro Recall,0.115591
5,Macro F1,0.133956
6,Hamming Loss,0.096429


### YOLO visual metrics

,image_id,gt_flags,predicted_flags,mean_iou,precision,recall
0,profile_000.jpg,clean,clean,0.437224,0.75,0.75
1,profile_001.jpg,clean,clean,0.576963,0.75,0.75
2,profile_002.jpg,Group_Photo_Ambiguity,Group_Photo_Ambiguity,0.099158,0.75,0.75
3,profile_003.jpg,Group_Photo_Ambiguity,Group_Photo_Ambiguity,0.145029,0.75,0.75
4,profile_004.jpg,Face_Obscured,No_Face_Present,0.000000,0.75,0.75
5,profile_005.jpg,Face_Obscured,No_Face_Present,0.000000,0.75,0.75
6,profile_006.jpg,No_Face_Present,No_Face_Present,1.000000,0.75,0.75
7,profile_007.jpg,No_Face_Present,No_Face_Present,1.000000,0.75,0.75


## 8. OpenRouter beautification layer

This sends the structured assessment to OpenRouter and asks an LLM to rewrite it into a polished report.

Do **not** hardcode your API key in the submitted notebook. Set it as an environment variable instead:

```python
os.environ["OPENROUTER_API_KEY"] = "your_key_here"
os.environ["OPENROUTER_MODEL"] = "openrouter/auto"
```

If no API key is set, the notebook uses a local fallback report so that the demo still runs.

In [8]:
def call_openrouter_for_report(assessment):
    api_key = os.getenv("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")
    model = os.getenv("OPENROUTER_MODEL", "openrouter/auto")

    if not api_key:
        return None

    system_prompt = (
        "You are a careful dating profile auditing assistant. "
        "Turn structured model outputs into a polished user-facing report. "
        "Stay grounded in the JSON. Do not invent new red flags. "
        "Use a balanced, non-judgmental tone and explain this is an automated prototype."
    )

    user_prompt = f"""
Beautify this raw dating profile audit into a clean report for a conceptual mobile app.

Required sections:
1. Overall audit result
2. Text/bio red flags
3. Tabular/profile metadata signals
4. Visual profile image signals
5. Model performance note with Precision, Recall, and IoU where available
6. Final recommendation to the app user

Raw assessment JSON:
{json.dumps(assessment, indent=2)}
""".strip()

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
            "X-Title": "Dating Red Flag Detector Notebook Demo",
        },
        json={
            "model": model,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            "temperature": 0.35,
        },
        timeout=60,
    )

    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]

## 9. Local fallback report

This produces a report without using the API. It is useful for testing or for submission screenshots when the API is unavailable.

In [9]:
def generate_local_report(assessment):
    risk_score = assessment["overall_risk_score"]
    risk_level_text = assessment["overall_risk_level"]
    text_flags = assessment["detected_text_tabular_flags"]
    visual_flags = assessment["detected_visual_flags"]
    combined_scores = assessment["combined_label_scores"]
    visual = assessment["module_outputs"]["visual_yolo"]

    lines = []
    lines.append(f"# Dating Profile Audit Report — Profile {assessment['profile_id']}")
    lines.append("")
    lines.append(f"**Overall result:** {risk_level_text} risk ({risk_score}/100).")
    lines.append("")
    lines.append("This is an automated prototype assessment. It should support user decision-making, not judge the person as inherently unsafe.")
    lines.append("")

    lines.append("## 1. Text and bio red flags")
    if text_flags:
        for flag in text_flags:
            info = combined_scores[flag]
            lines.append(f"- **{flag}** — {info['description']} Mean ensemble probability: {info['mean_probability']:.4f}.")
    else:
        lines.append("- No major text/tabular red flags were detected by the ensemble threshold.")
    lines.append("")

    lines.append("## 2. Tabular/profile metadata signals")
    tabular_output = assessment["module_outputs"]["tabular_smote_random_forest"]
    if tabular_output["detected_flags"]:
        for flag in tabular_output["detected_flags"]:
            score = tabular_output["label_scores"][flag]["probability"]
            lines.append(f"- **{flag}** was triggered by the tabular SMOTE model with probability {score:.4f}.")
    else:
        lines.append("- The tabular SMOTE model did not independently trigger any metadata flag.")
    lines.append("")

    lines.append("## 3. Visual profile image signals")
    if visual.get("available"):
        for flag in visual_flags:
            explanation = visual.get("visual_flag_explanations", {}).get(flag, "")
            lines.append(f"- **{flag}** — {explanation}")
        lines.append(f"- Detected persons: {visual.get('num_persons_detected')}")
        lines.append(f"- IoU for this row: {visual.get('iou_metric')}")
        lines.append(f"- Visual source: {visual.get('visual_flag_source')}")
    else:
        lines.append(f"- Visual module unavailable: {visual.get('reason')}")
    lines.append("")

    lines.append("## 4. Model performance note")
    perf = assessment.get("previous_model_performance", {})

    if "nlp_tfidf_logreg_overall" in perf:
        nlp_text = ", ".join([f"{row['Metric']}: {row['Score']:.4f}" for row in perf["nlp_tfidf_logreg_overall"]])
        lines.append(f"- NLP model overall metrics: {nlp_text}.")

    if "tabular_smote_random_forest_overall" in perf:
        tab_text = ", ".join([f"{row['Metric']}: {row['Score']:.4f}" for row in perf["tabular_smote_random_forest_overall"]])
        lines.append(f"- Tabular SMOTE model overall metrics: {tab_text}.")

    if "yolo_visual_subset" in perf:
        yolo = perf["yolo_visual_subset"]
        lines.append(f"- YOLO visual subset metrics: Precision {yolo.get('mean_precision')}, Recall {yolo.get('mean_recall')}, Mean IoU {yolo.get('mean_iou')}.")

    if not perf:
        lines.append("- Metric CSV files were not found, so performance metrics were not attached.")
    lines.append("")

    lines.append("## 5. Final recommendation")
    if risk_score >= 70:
        lines.append("- The app should warn the user to review this profile carefully before engaging.")
    elif risk_score >= 40:
        lines.append("- The app should show a caution note and highlight the specific evidence.")
    else:
        lines.append("- The app can mark the profile as relatively low-risk while still showing transparent model evidence.")

    return "\n".join(lines)

# ============================================================
# CUSTOM DEMO INPUT CELL
# Run this cell when you want to test your own dating profile.
# Make sure previous cells in Notebook 05 have already loaded:
# df, models, label_columns, run_nlp_inference, run_tabular_inference,
# combine_label_scores, risk_level, load_metrics_summary,
# generate_local_report, call_openrouter_for_report
# ============================================================

CUSTOM_PROFILE_ID = "custom_demo_001"

# 1. Edit this part with your own demo profile data
custom_profile = {
    # Main text used by the NLP model
    "full_bio": """
    I'm not like everyone else on here. I am always right. Silent treatment is a must for the other party to learn from their mistakes on their own. It is actually frustrating when people do not know my interest because it shows that they are not observants. I keep distance with people who has wronged me because i do not feel like they still need to be in my life once they hurt my feelings.
    """,

    # Clean bio can be same as full_bio for demo input
    "clean_bio": """
     I'm not like everyone else on here. I am always right. Silent treatment is a must for the other party to learn from their mistakes on their own. It is actually frustrating when people do not know my interest because it shows that they are not observants. I keep distance with people who has wronged me because i do not feel like they still need to be in my life once they hurt my feelings.
    """,

    # Metadata used by the tabular SMOTE model
    "age": 50,
    "status": "single",
    "sex": "m",
    "orientation": "straight",
    "body_type": "average",
    "diet": "vegan",
    "drinks": "often",
    "drugs": "often",
    "education": "graduated from high school",
    "ethnicity": "white",
    "height": 150,
    "income": 10000,
    "job": "CEO",
    "offspring": "had kids",
    "pets": "likes dogs",
    "religion": "agnosticism",
    "sign": "gemini",
    "smokes": "often"
}

# 2. Optional visual demo input
# Since this cell does not run YOLO on a real image, this is a manual/conceptual visual result.
# You can change this to "clean", "No_Face_Present", "Group_Photo_Ambiguity", or "Face_Obscured".
custom_visual_flags = ["clean"]

custom_visual_assessment = {
    "available": True,
    "module": "Manual visual demo input",
    "detected_visual_flags": custom_visual_flags,
    "visual_risk_score": 0.0 if custom_visual_flags == ["clean"] else 0.65,
    "visual_flag_explanations": {
        flag: visual_flag_descriptions.get(flag, "Custom visual flag entered for demo.")
        for flag in custom_visual_flags
    },
    "bounding_boxes": [],
    "confidence_scores": [],
    "num_persons_detected": 1,
    "iou_metric": None,
    "visual_flag_source": "manual_demo_input",
    "image_path": "custom_demo_no_image"
}


# 3. Build assessment from custom input
def build_custom_agent_assessment(custom_profile, visual_assessment=None, profile_id="custom_demo"):
    profile_text = custom_profile.get("clean_bio", custom_profile.get("full_bio", ""))

    nlp = run_nlp_inference(profile_text)
    transformer = run_transformer_inference(profile_text)
    tabular = run_tabular_inference(custom_profile)

    if visual_assessment is None:
        visual_assessment = {
            "available": False,
            "reason": "No visual input was provided for this custom demo.",
            "detected_visual_flags": [],
            "visual_risk_score": 0.0
        }

    combined_scores = combine_label_scores(nlp, tabular, transformer)
    detected_text_tabular_flags = [
        label for label, info in combined_scores.items()
        if info["ensemble_predicted"] == 1
    ]

    label_probabilities = [
        info["mean_probability"]
        for info in combined_scores.values()
    ]

    top_label_risk = (
        float(np.mean(sorted(label_probabilities, reverse=True)[:3]))
        if label_probabilities else 0.0
    )

    visual_risk = visual_assessment.get("visual_risk_score", 0.0)

    overall_score = round((0.70 * top_label_risk + 0.30 * visual_risk) * 100, 2)

    return {
        "profile_id": profile_id,
        "profile_summary": {
            field: custom_profile.get(field, None)
            for field in profile_display_fields
        },
        "bio_excerpt": str(custom_profile.get("full_bio", ""))[:1200],
        "overall_risk_score": overall_score,
        "overall_risk_level": risk_level(overall_score),
        "detected_text_tabular_flags": detected_text_tabular_flags,
        "detected_visual_flags": visual_assessment.get("detected_visual_flags", []),
        "combined_label_scores": combined_scores,
        "module_outputs": {
            "nlp_tfidf_logreg": nlp,
            "nlp_transformer": transformer,
            "tabular_smote_random_forest": tabular,
            "visual_yolo": visual_assessment,
        },
        "previous_model_performance": load_metrics_summary()
    }


custom_assessment = build_custom_agent_assessment(
    custom_profile=custom_profile,
    visual_assessment=custom_visual_assessment,
    profile_id=CUSTOM_PROFILE_ID
)

# 4. Try OpenRouter beautification first. If no API key is found, use local fallback report.
try:
    custom_llm_report = call_openrouter_for_report(custom_assessment)
except Exception as e:
    print("OpenRouter call failed. Using local fallback report instead.")
    print("Reason:", e)
    custom_llm_report = None

custom_final_report = custom_llm_report if custom_llm_report else generate_local_report(custom_assessment)

# 5. Display result
display(Markdown(custom_final_report))

# 6. Save result
custom_assessment_path = REPORTS_DIR / "custom_demo_assessment.json"
custom_report_path = REPORTS_DIR / "custom_demo_report.md"

with open(custom_assessment_path, "w", encoding="utf-8") as f:
    json.dump(custom_assessment, f, indent=2)

with open(custom_report_path, "w", encoding="utf-8") as f:
    f.write(custom_final_report)

print("Saved custom raw assessment to:", custom_assessment_path)
print("Saved custom final report to:", custom_report_path)

## 10. Generate and save final report

In [10]:
try:
    llm_report = call_openrouter_for_report(raw_assessment)
except Exception as e:
    print("OpenRouter call failed. Using local fallback report instead.")
    print("Reason:", e)
    llm_report = None

final_report = llm_report if llm_report else generate_local_report(raw_assessment)

display(Markdown(final_report))

assessment_path = REPORTS_DIR / "agent_sample_assessment.json"
report_path = REPORTS_DIR / "agent_sample_report.md"

with open(assessment_path, "w", encoding="utf-8") as f:
    json.dump(raw_assessment, f, indent=2)

with open(report_path, "w", encoding="utf-8") as f:
    f.write(final_report)

print("Saved raw assessment to:", assessment_path)
print("Saved final report to:", report_path)

# Dating Profile Audit Report — Profile 0

**Overall result:** Low-to-Moderate risk (26.95/100).

This is an automated prototype assessment. It should support user decision-making, not judge the person as inherently unsafe.

## 1. Text and bio red flags
- **sarcasm_cynicism** — Sarcastic or dismissive tone that may need careful interpretation. Mean ensemble probability: 0.5980.

## 2. Tabular/profile metadata signals
- **sarcasm_cynicism** was triggered by the tabular SMOTE model with probability 0.7700.

## 3. Visual profile image signals
- **clean** — No major visual presentation issue detected.
- Detected persons: 1
- IoU for this row: 0.4372
- Visual source: yolo_live

## 4. Model performance note
- NLP model overall metrics: Micro Precision: 0.4898, Micro Recall: 0.4103, Micro F1: 0.4465, Macro Precision: 0.3468, Macro Recall: 0.3268, Macro F1: 0.3092, Hamming Loss: 0.1417.
- Tabular SMOTE model overall metrics: Micro Precision: 0.8600, Micro Recall: 0.3675, Micro F1: 0.5150, Macro Precision: 0.1593, Macro Recall: 0.1156, Macro F1: 0.1340, Hamming Loss: 0.0964.
- YOLO visual subset metrics: Precision 0.75, Recall 0.75, Mean IoU 0.4073.

## 5. Final recommendation
- The app can mark the profile as relatively low-risk while still showing transparent model evidence.

Saved raw assessment to: C:\Users\LENOVO\anaconda3\Red-Flagger\reports\agent_sample_assessment.json
Saved final report to: C:\Users\LENOVO\anaconda3\Red-Flagger\reports\agent_sample_report.md


## 11. Conceptual mobile interface mockup

This is not a real app yet. It is a visual prototype of how the auditor result could appear in the final mobile interface deliverable.

In [13]:
def render_mobile_auditor_card(assessment):
    risk_score = assessment["overall_risk_score"]
    risk_level_text = assessment["overall_risk_level"]
    text_flags = assessment["detected_text_tabular_flags"]
    visual_flags = assessment["detected_visual_flags"]

    flag_html = "".join(
        f"<span class='pill'>{html.escape(flag)}</span>"
        for flag in (text_flags + visual_flags)
    ) or "<span class='pill clean'>No major flags</span>"

    bio_excerpt = html.escape(str(assessment.get("bio_excerpt", ""))[:350])

    card = f"""
    <style>
        .phone-frame {{
            width: 360px;
            border-radius: 32px;
            padding: 18px;
            background: #111827;
            color: #f9fafb;
            font-family: Arial, sans-serif;
            box-shadow: 0 12px 30px rgba(0,0,0,0.25);
        }}
        .phone-card {{
            background: #1f2937;
            border-radius: 24px;
            padding: 18px;
            border: 1px solid #374151;
        }}
        .small {{ color: #9ca3af; font-size: 12px; }}
        .score {{ font-size: 42px; font-weight: bold; margin: 8px 0; }}
        .level {{ font-size: 16px; margin-bottom: 12px; }}
        .pill {{
            display: inline-block;
            padding: 6px 10px;
            border-radius: 999px;
            margin: 4px 4px 4px 0;
            background: #374151;
            font-size: 12px;
        }}
        .clean {{ background: #065f46; }}
        .section {{ margin-top: 16px; }}
    </style>
    <div class="phone-frame">
        <div class="phone-card">
            <div class="small">Dating Red Flag Detector</div>
            <h2>Profile Audit</h2>
            <div class="small">Profile ID: {assessment['profile_id']}</div>
            <div class="score">{risk_score}</div>
            <div class="level">{html.escape(risk_level_text)} Risk</div>
            <div class="section">
                <div class="small">Detected Signals</div>
                {flag_html}
            </div>
            <div class="section">
                <div class="small">Bio Preview</div>
                <p>{bio_excerpt}...</p>
            </div>
            <div class="section small">
                Tap to view model evidence, Precision/Recall, and YOLO IoU details.
            </div>
        </div>
    </div>
    """
    return HTML(card)

display(render_mobile_auditor_card(raw_assessment))

## 12. What Notebook 05 demonstrates

This notebook connects the full assignment pipeline:

- **OkCupid profile reader:** selects and displays one cleaned profile row.
- **NLP model:** audits written bio red flags, including sarcasm/cynicism.
- **Transformer support:** optionally uses the DistilBERT model from Notebook 02.
- **SMOTE tabular model:** audits imbalanced metadata-based signals.
- **YOLO visual module:** attaches visual flags, bounding boxes, confidence scores, and IoU.
- **LLM orchestration:** uses OpenRouter to beautify the structured assessment.
- **Conceptual mobile interface:** shows how the audit could appear to an end-user.